# 05 · Modelling: baseline, models, metric, choice, test and the final prediction

**Task:** for each quarter, rank all styles and pick the three that will be most successful.

**Setup**
- Rows: one per (style, quarter), for styles that sold at least once before the quarter. Features use only data from
  before the quarter starts (`data_science/features/build.py`). Target: the quarter's success score.
- **Business rule:** only *commercially relevant* styles can be recommended: styles whose revenue last quarter was in the
  top 10% of styles that sold. It uses only pre-quarter data, so it also applies to the final prediction. All results
  below are among commercially relevant styles: our picks and the true winners both come from this pool.
- **Walk-forward:** each evaluated quarter is predicted by models trained on **all earlier quarters**.
  - Validation: **Autumn 2019, Winter 2019**. Used to choose the metric and the model.
  - Test: **Spring 2020, Summer 2020**. Used only to report the chosen model; the validation quarters are part of the
    training data for these.
- Final model: retrained on **all** quarters and used to predict **Autumn 2020**.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_science").is_dir())
sys.path.insert(0, str(ROOT))

import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from data_science.features.build import CATEGORICAL_FEATURES, FEATURES, NUMERIC_FEATURES, TARGET
from data_science.train import load_or_build_features

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_colwidth", 60)

features = load_or_build_features()
labelled = features[features[TARGET].notna()]
VALIDATION, TEST = ["2019_autumn", "2019_winter"], ["2020_spring", "2020_summer"]
print(f"{len(labelled):,} labelled style-quarters, {len(FEATURES)} features; "
      f"{(features[TARGET].isna()).sum():,} styles to predict for {features.loc[features[TARGET].isna(), 'quarter_id'].iloc[0]}")

122,475 labelled style-quarters, 35 features; 22,995 styles to predict for 2020_autumn


## 1. The business rule

In [2]:
COMMERCIAL_TOP_SHARE = 0.10

def commercially_relevant(df, top_share=COMMERCIAL_TOP_SHARE):
    """True for styles whose revenue last quarter is in the top `top_share` of styles that sold (per quarter)."""
    revenue = np.expm1(df.log_revenue_lag1)
    cutoff = revenue.where(revenue > 0).groupby(df.quarter_id, observed=True).transform(lambda r: r.quantile(1 - top_share))
    return revenue >= cutoff

print("commercially relevant styles per quarter:")
print(labelled[commercially_relevant(labelled)].groupby("quarter_id", observed=True).size().to_string())

commercially relevant styles per quarter:
quarter_id
2018_winter    1246
2019_autumn    1274
2019_spring    1293
2019_summer    1301
2019_winter    1212
2020_spring    1258
2020_summer    1200


## 2. Baseline: last quarter's success score

The baseline predicts that each style will be exactly as successful as it was last quarter.

- It is the natural "do nothing" forecast: a merchandiser without a model would repeat last quarter's winners.
- It is a strong bar. Success persists from one quarter to the next (rank correlation below), so a model only earns
  its place if it beats persistence.
- It uses the same information that is available at decision time, so the comparison is fair.

In [3]:
corr = labelled.groupby("quarter_id", observed=True).apply(
    lambda d: spearmanr(d[TARGET], d.score_lag1).statistic).rename("rank correlation with last quarter")
corr.round(2).to_frame()

,rank correlation with last quarter
quarter_id,
2018_winter,0.78
2019_autumn,0.69
2019_spring,0.68
2019_summer,0.78
2019_winter,0.78
2020_spring,0.72
2020_summer,0.79


## 3. Training code

Two models, both trained on **log(1 + success score)** (the target is very skewed and half of it is zero) and
ranked by their prediction:

- **Ridge regression**: linear; one-hot encoded attributes, standardised numeric features, missing values
  imputed with a missing-value flag. Tests how far a linear relationship goes.
- **LightGBM** (gradient-boosted trees): non-linear, with interactions and native categorical handling. Averaged over
  5 random seeds so that its top picks don't depend on one seed.

In [4]:
def baseline_last_quarter(train, test):
    return test.score_lag1.to_numpy()


def as_model_input(df):
    return df[FEATURES].astype({c: str for c in CATEGORICAL_FEATURES})


def make_ridge():
    pre = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                          ("scale", StandardScaler())]), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CATEGORICAL_FEATURES),
    ])
    return Pipeline([("pre", pre), ("ridge", Ridge(alpha=1.0))])


def ridge(train, test):
    model = make_ridge().fit(as_model_input(train), np.log1p(train[TARGET]))
    return model.predict(as_model_input(test))


LGBM_PARAMS = dict(n_estimators=600, learning_rate=0.03, num_leaves=63, min_child_samples=100,
                   subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, verbose=-1)
SEEDS = [1, 2, 3, 4, 5]

def lightgbm(train, test):
    preds = []
    for seed in SEEDS:
        m = lgb.LGBMRegressor(objective="regression", random_state=seed, **LGBM_PARAMS)
        m.fit(train[FEATURES], np.log1p(train[TARGET]), categorical_feature=CATEGORICAL_FEATURES)
        preds.append(m.predict(test[FEATURES]))
    return np.mean(preds, axis=0)


MODELS = {"baseline: last quarter": baseline_last_quarter, "ridge": ridge, "lightgbm": lightgbm}

## 4. Metrics

All metrics are computed among commercially relevant styles, comparing the predicted ranking with the actual success scores.

| Metric | Meaning |
|---|---|
| **top-3 quality** | average actual score of our 3 picks ÷ average actual score of the true top 3 (1.0 = best possible picks) |
| top-10 / top-50 quality | the same for the top 10 / top 50 |
| NDCG@50 | quality of the order of the top 50, weighting higher positions more |
| Spearman | rank correlation over all commercially relevant styles |

In [5]:
def top_k_quality(y, p, k):
    return y[np.argsort(-p, kind="stable")[:k]].mean() / np.sort(y)[::-1][:k].mean()

def ndcg(y, p, k=50):
    discount = 1 / np.log2(np.arange(2, k + 2))
    return (y[np.argsort(-p, kind="stable")[:k]] * discount).sum() / (np.sort(y)[::-1][:k] * discount).sum()

def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return {"top-3 quality": top_k_quality(y, p, 3), "top-10 quality": top_k_quality(y, p, 10),
            "top-50 quality": top_k_quality(y, p, 50), "NDCG@50": ndcg(y, p), "Spearman": spearmanr(y, p).statistic}


def walk_forward(quarters, models):
    rows = []
    for q in quarters:
        qi = labelled.loc[labelled.quarter_id == q, "qi"].iloc[0]
        train, test = labelled[labelled.qi < qi], labelled[labelled.qi == qi]
        keep = commercially_relevant(test).to_numpy()
        for name in models:
            pred = np.asarray(MODELS[name](train, test), float)
            rows.append({"quarter": q, "model": name, "trained on": f"{train.quarter_id.nunique()} quarters",
                         **metrics(test[TARGET].to_numpy()[keep], pred[keep])})
    return pd.DataFrame(rows)

## 5. Validation: Autumn 2019 and Winter 2019

In [6]:
val = walk_forward(VALIDATION, MODELS)
val.round(3)

,quarter,model,trained on,top-3 quality,top-10 quality,top-50 quality,NDCG@50,Spearman
0,2019_autumn,baseline: last quarter,3 quarters,0.858,0.896,0.887,0.883,0.713
1,2019_autumn,ridge,3 quarters,0.911,0.975,0.956,0.940,0.833
2,2019_autumn,lightgbm,3 quarters,0.994,0.935,0.938,0.942,0.828
3,2019_winter,baseline: last quarter,4 quarters,0.869,0.962,0.962,0.960,0.817
4,2019_winter,ridge,4 quarters,0.949,0.985,0.960,0.963,0.869
5,2019_winter,lightgbm,4 quarters,0.899,0.959,0.962,0.943,0.863


In [7]:
val_mean = val.groupby("model", sort=False)[["top-3 quality", "top-10 quality", "top-50 quality", "NDCG@50", "Spearman"]].mean()
val_mean.round(3)

,top-3 quality,top-10 quality,top-50 quality,NDCG@50,Spearman
model,,,,,
baseline: last quarter,0.863,0.929,0.925,0.922,0.765
ridge,0.930,0.980,0.958,0.951,0.851
lightgbm,0.946,0.947,0.950,0.943,0.846


## 6. Choosing the metric: top-3 quality

- It measures exactly the decision: **which three styles to produce**.
- It gives credit for near-misses: picking the 4th-best style instead of the 3rd is barely penalised, unlike counting exact hits.
- It reads simply: "our picks achieved X% of the best possible picks".

The other metrics describe the rest of the ranking (top 10, top 50, all styles), which is useful context but not the
decision. Top-3 quality is noisy because it rests on three styles per quarter, so it is averaged over quarters and read
together with the others.

In [8]:
top3 = val.pivot(index="model", columns="quarter", values="top-3 quality").loc[list(MODELS)]
top3["mean"] = top3.mean(axis=1)
top3["worst quarter"] = top3[VALIDATION].min(axis=1)
top3["gain over baseline (mean)"] = top3["mean"] - top3.loc["baseline: last quarter", "mean"]
top3.round(3)

quarter,2019_autumn,2019_winter,mean,worst quarter,gain over baseline (mean)
model,,,,,
baseline: last quarter,0.858,0.869,0.863,0.858,0.000
ridge,0.911,0.949,0.930,0.911,0.066
lightgbm,0.994,0.899,0.946,0.899,0.083


## 7. Choosing the model: ridge

Both models beat the baseline in both validation quarters. Between them:

- On the **mean**, LightGBM is slightly ahead, but the difference is small next to the noise of three picks per quarter,
  and LightGBM's whole lead comes from **one quarter** (Autumn 2019). It loses Winter 2019.
- Ridge has the **better worst quarter**: it is the more reliable choice when each quarter's decision matters.
- Ridge is **deterministic**: the same data always gives the same three picks. LightGBM's picks move with the random seed
  (shown below), which is why it needs seed averaging.
- Ridge's explanations are **exact**: a prediction is the sum of its feature contributions, which is what the API shows
  as "why this style".
- On **every other metric** ridge is ahead: top-10 quality 0.980 vs 0.947, top-50 0.958 vs 0.950, NDCG@50 0.951 vs 0.943,
  Spearman 0.851 vs 0.846. LightGBM's advantage exists only in the noisiest metric (top-3 mean).
- A linear model captures most of the signal: persistence of recent success plus seasonality.

**Chosen: ridge.** The choice is fixed here, before looking at the test quarters.

In [9]:
# LightGBM's sensitivity to the random seed (single seeds, top-3 quality, validation quarters)
rows = []
for q in VALIDATION:
    qi = labelled.loc[labelled.quarter_id == q, "qi"].iloc[0]
    train, test = labelled[labelled.qi < qi], labelled[labelled.qi == qi]
    keep = commercially_relevant(test).to_numpy()
    for seed in SEEDS:
        m = lgb.LGBMRegressor(objective="regression", random_state=seed, **LGBM_PARAMS)
        m.fit(train[FEATURES], np.log1p(train[TARGET]), categorical_feature=CATEGORICAL_FEATURES)
        rows.append({"quarter": q, "seed": seed, "top-3 quality": top_k_quality(test[TARGET].to_numpy()[keep], m.predict(test[FEATURES])[keep], 3)})
seeds = pd.DataFrame(rows).pivot(index="seed", columns="quarter", values="top-3 quality")
seeds.round(3)

quarter,2019_autumn,2019_winter
seed,,
1,0.994,0.982
2,0.994,0.832
3,0.994,0.899
4,1.000,0.899
5,0.994,0.899


## 8. Test: Spring 2020 and Summer 2020, ridge vs the baseline

In [10]:
test = walk_forward(TEST, ["baseline: last quarter", "ridge"])
test.round(3)

,quarter,model,trained on,top-3 quality,top-10 quality,top-50 quality,NDCG@50,Spearman
0,2020_spring,baseline: last quarter,5 quarters,0.819,0.816,0.882,0.870,0.698
1,2020_spring,ridge,5 quarters,0.955,0.938,0.936,0.934,0.884
2,2020_summer,baseline: last quarter,6 quarters,0.992,0.956,0.937,0.942,0.821
3,2020_summer,ridge,6 quarters,0.992,0.948,0.965,0.958,0.881


In [11]:
test_mean = test.groupby("model", sort=False)[["top-3 quality", "top-10 quality", "top-50 quality", "NDCG@50", "Spearman"]].mean()
test_mean.loc["ridge − baseline"] = test_mean.loc["ridge"] - test_mean.loc["baseline: last quarter"]
test_mean.round(3)

,top-3 quality,top-10 quality,top-50 quality,NDCG@50,Spearman
model,,,,,
baseline: last quarter,0.906,0.886,0.909,0.906,0.759
ridge,0.973,0.943,0.950,0.946,0.882
ridge − baseline,0.068,0.057,0.041,0.040,0.123


- Ridge beats the baseline clearly in **Spring 2020** (the COVID quarter, when last quarter's pattern broke) and
  matches it in **Summer 2020**, where the true winners were exactly last quarter's leaders and the baseline is already
  near the maximum.
- Over the four evaluated quarters (validation and test), ridge beats the baseline in three and ties in one.

## 9. Final model: retrain on all quarters and predict Autumn 2020

In [12]:
final = make_ridge().fit(as_model_input(labelled), np.log1p(labelled[TARGET]))
print("trained on:", ", ".join(labelled.sort_values("qi").quarter_id.unique()))

upcoming = features[features[TARGET].isna()].copy()
upcoming["prediction_score"] = np.expm1(final.predict(as_model_input(upcoming)))
upcoming["commercially_relevant"] = commercially_relevant(upcoming)
ranked = upcoming[upcoming.commercially_relevant].sort_values("prediction_score", ascending=False)
ranked.insert(0, "rank", range(1, len(ranked) + 1))
print(f"{upcoming.quarter_id.iloc[0]}: {len(upcoming):,} styles scored, {len(ranked):,} commercially relevant")
ranked.head(10)[["rank", "style_name", "prediction_score", "score_lag1", "score_lag4", "score_last4w", "score_mean4"]].round(3)

trained on: 2018_winter, 2019_spring, 2019_summer, 2019_autumn, 2019_winter, 2020_spring, 2020_summer
2020_autumn: 22,995 styles scored, 1,272 commercially relevant


,rank,style_name,prediction_score,score_lag1,score_lag4,score_last4w,score_mean4
101936,1,Ladieswear · Skirt · Solid · Black · Dark,6.627,4.474,3.340,3.971,3.659
85577,2,Ladieswear · Dress · Solid · Black · Dark,5.577,3.741,3.102,3.411,3.395
66986,3,Divided · Sweater · Solid · Black · Dark,5.349,3.272,4.125,2.911,3.785
106752,4,Ladieswear · Sweater · Solid · Black · Dark,5.188,3.190,3.952,2.819,3.684
132237,5,Menswear · Sweater · Solid · Black · Dark,5.103,3.493,3.406,3.121,3.330
127827,6,Menswear · Shirt · Solid · White · Light,4.232,3.397,3.020,3.071,3.198
76892,7,Ladieswear · Blazer · Solid · Black · Dark,4.225,3.406,2.511,3.023,2.986
55822,8,Divided · Dress · Solid · Black · Dark,3.838,3.115,2.699,2.992,2.775
114951,9,Ladieswear · Trousers · Solid · Black · Dark,3.784,3.287,2.663,2.932,3.000
75903,10,Ladieswear · Bikini top · Solid · Black · Dark,3.764,3.121,2.842,3.075,2.801


### Why these three: feature contributions

Ridge is linear in log(1 + success score), so each prediction is exactly the average prediction plus the sum of
contributions, coefficient × (feature value − training average).

In [13]:
pre, coef = final.named_steps["pre"], final.named_steps["ridge"].coef_
train_mean = np.asarray(pre.transform(as_model_input(labelled)).mean(axis=0)).ravel()

def original_feature(name):
    kind, name = name.split("__", 1)
    return name.removeprefix("missingindicator_") if kind == "num" else next(c for c in CATEGORICAL_FEATURES if name.startswith(c + "_"))

names = [original_feature(n) for n in pre.get_feature_names_out()]
top3_rows = ranked.head(3)
x = pre.transform(as_model_input(top3_rows))
x = x.toarray() if hasattr(x, "toarray") else np.asarray(x)
contrib = pd.DataFrame((x - train_mean) * coef, columns=names).T.groupby(level=0).sum().T
contrib.index = top3_rows.style_name

for style, c in contrib.iterrows():
    top = c.reindex(c.abs().sort_values(ascending=False).index).head(5)
    print(f"\n{style}  (predicted score {top3_rows.set_index('style_name').loc[style, 'prediction_score']:.2f})")
    for feat, val in top.items():
        print(f"   {feat:28s} {val:+.3f}")


Ladieswear · Skirt · Solid · Black · Dark  (predicted score 6.63)
   score_last4w                 +1.173
   score_mean4                  +0.534
   score_lag1                   +0.142
   score_max4                   -0.109
   score_lag4                   +0.089

Ladieswear · Dress · Solid · Black · Dark  (predicted score 5.58)
   score_last4w                 +1.003
   score_mean4                  +0.494
   new_articles_13w             +0.238
   articles_lag1                -0.168
   score_lag1                   +0.118

Divided · Sweater · Solid · Black · Dark  (predicted score 5.35)
   score_last4w                 +0.851
   score_mean4                  +0.553
   score_lag4                   +0.111
   score_max4                   -0.109
   score_lag3                   +0.104


## 10. Summary

| | |
|---|---|
| Baseline | last quarter's success score |
| Metric | top-3 quality among commercially relevant styles |
| Validation (Autumn + Winter 2019) | ridge 0.930, LightGBM 0.946, baseline 0.863 → **ridge chosen** for its better worst quarter, stability and exact explanations |
| Test (Spring + Summer 2020) | ridge **0.973** vs baseline 0.906 top-3 quality; ridge better on every metric (top-10 0.943 vs 0.886, NDCG@50 0.946 vs 0.906) |
| **Autumn 2020 top 3** | 1. Ladieswear · Skirt · Solid · Black · Dark<br>2. Ladieswear · Dress · Solid · Black · Dark<br>3. Divided · Sweater · Solid · Black · Dark |

The predicted score is on the success-score scale; values above the historical maximum come from linear extrapolation,
and only the ranking is used.